# 🎬 VideoFX v4 — gerador rápido de vídeos (estilo MoneyPrinter)

Roda 100% no navegador: narração TTS neural em português, imagens de fundo, legendas sincronizadas, música e efeito Ken Burns.

**v4** — roteiro IA em cadeia (Gemini→OpenRouter→HF), voz ElevenLabs→Piper, imagens Pollinations→Horde→Pixabay, trechos do YouTube.  
**Colab:** `Ambiente de execução → Executar tudo` *(ou clique nos botões ▶ de cada célula)*  
**Kaggle:** *File → Import Notebook → from GitHub* + Settings → **Internet: ON**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jushirambo/videofx/blob/main/videofx_colab.ipynb)


## 1️⃣ Instalação (~1 min)

⚠️ **Importante:** quando o Colab perguntar "Acessar secrets?", clique **Conceder/Grant access** — é isso que injeta GEMINI_API_KEY, ELEVEN_LABS_API_KEY, etc. no ambiente.


In [ ]:
%%capture
!git clone https://github.com/Jushirambo/videofx.git /content/videofx
%cd /content/videofx
!pip install -q -r requirements.txt
import os, sys
sys.path.insert(0, '/content/videofx')
try:
    from google.colab import userdata
    for name in ["GEMINI_API_KEY", "GEMINI_API_KEY_1", "OPENROUTER_KEY", "HF_TOKEN",
                 "ELEVEN_LABS_API_KEY", "PIXABAY_KEY", "POLINATIONS", "AIHORDE"]:
        try:
            os.environ.setdefault(name, userdata.get(name))
        except Exception:
            pass
    print("🔑 secrets do Colab carregados:", [k for k in os.environ if k in
          ("GEMINI_API_KEY", "GEMINI_API_KEY_1", "OPENROUTER_KEY", "HF_TOKEN",
           "ELEVEN_LABS_API_KEY", "PIXABAY_KEY", "POLINATIONS", "AIHORDE")])
except ImportError:
    print("Kaggle: exporte as keys com os.environ manualmente")
print('✅ pronto')


## 2️⃣ Configure o vídeo
Edite os campos abaixo e rode a célula (▶). Os **secrets do Colab** (🔑 painel direito) são carregados automaticamente na instalação.

**Roteiro via IA (cadeia de fallbacks):** escolha `provider = auto` para tentar **Gemini → OpenRouter → HuggingFace** (usa a chave que existir em Secrets) e cair no template local se nenhum responder. Ou force um provedor específico.

**Trechos do YouTube (opcional):** preencha `yt_clips` com uma linha por cena, formato `URL|segundo_inicial` (ex.: `https://youtube.com/watch?v=abc|12`) ou `termo de busca|12`. Se o download falhar, a cena volta para imagens IA automaticamente. Ou marque `yt_auto` para buscar clipes automaticamente a partir do tema.

Ou escreva suas falas em `ROTULO`, uma por cena.


In [ ]:
#@title ⚙️ Configuração do vídeo { display-mode: "form" }
topic = "Fatos incríveis sobre o universo" #@param {type:"string"}
title = "" #@param {type:"string"}
n_scenes = 5 #@param {type:"slider", min:2, max:8, step:1}
aspect = "9:16" #@param ["9:16", "16:9", "1:1"]
#@markdown ---
#@markdown **Roteiro IA** (cadeia com fallbacks; `auto` usa Gemini→OpenRouter→HF conforme a chave disponível)
provider = "auto" #@param ["auto", "template", "openrouter", "huggingface", "gemini"]
model = "" #@param {type:"string"}
api_key = "" #@param {type:"string"}
script_style = "viral" #@param ["viral", "educativo", "storytelling"]
#@markdown ---
#@markdown **Voz** — ElevenLabs só em plano pago (contas grátis não usam library voices via API); sem plano, Piper pt-BR neural assume
tts_providers = "elevenlabs,piper" #@param {type:"string"}
elevenlabs_voice = "rachel" #@param ["rachel", "drew", "bella", "gabriel", "michele"]
#@markdown ---
#@markdown **Trechos do YouTube** (uma linha por cena: `URL|segundo` ou `busca|segundo`; falha → imagem IA)
yt_clips = "" #@param {type:"string"}
yt_auto = False #@param {type:"boolean"}
 #@markdown ---
#@markdown **Efeitos visuais**
transitions = "fade,slideleft,zoomin,smoothleft,circleopen" #@param {type:"string"}
transition_duration = 0.5 #@param {type:"slider", min:0.1, max:1.2, step:0.1}
subtitle_style = "pop" #@param ["pop", "box", "clean"]
music_mood = "calm" #@param ["calm", "energetic"]
particles = True #@param {type:"boolean"}
shine = True #@param {type:"boolean"}
vignette = True #@param {type:"boolean"}
color_boost = True #@param {type:"boolean"}
progress_bar = True #@param {type:"boolean"}
ROTULO = "" #@param {type:"string"}

import json, os

def split_env_list(s, default):
    return [x.strip() for x in s.split(",") if x.strip()] or default

cfg = {
    "topic": topic,
    "aspect": aspect,
    "provider": provider,
    "model": model or None,
    "api_key": api_key or os.environ.get("OPENROUTER_KEY") or os.environ.get("OPENROUTER_API_KEY"),
    "script_style": script_style,
    "tts_providers": split_env_list(tts_providers, ["elevenlabs", "piper"]),
    "elevenlabs_voice": elevenlabs_voice,
    "yt_clips": [l.strip() for l in yt_clips.splitlines() if l.strip()] or None,
    "yt_auto": yt_auto,
    "transitions": transitions,
    "transition_duration": transition_duration,
    "subtitle_style": subtitle_style,
    "music_mood": music_mood,
    "particles": particles,
    "shine": shine,
    "vignette": vignette,
    "color_boost": color_boost,
    "progress_bar": progress_bar,
    "allow_ai_images": True,
    "output_name": "video_final.mp4",
}
if not title.strip():
    cfg.pop("title", None)
if provider == "template":
    cfg["n_scenes"] = n_scenes
if ROTULO.strip():
    cfg["scenes"] = [l.strip() for l in ROTULO.splitlines() if l.strip()]
json.dump(cfg, open("config.json", "w"), ensure_ascii=False, indent=2)
extra = " | " + str(len(cfg["yt_clips"])) + " clipes do YouTube" if cfg["yt_clips"] else ""
print("config salva ✅", 
      "(roteiro manual)" if ROTULO.strip() else "(roteiro IA: " + provider + ")", extra)


## 3️⃣ Gere o vídeo (~2–4 min)
 imagem + narracão + música + legendas são geradas nesta célula.


In [ ]:
!python main.py --config config.json


## 4️⃣ Assista e baixe
O vídeo aparece abaixo e um download é disparado no navegador (no Kaggle, use o painel *Output*).


In [ ]:
from IPython.display import Video, display
display(Video("out/video_final.mp4", embed=True, height=560, width=320))
try:
    from google.colab import files
    files.download("out/video_final.mp4")
except ImportError:
    print("Kaggle: baixe em Data ▸ Output ▸ video_final.mp4"))


## 🎛️ (Opcional) Interface interativa no navegador
Rode para abrir um painel Gradio com link público (`*.gradio.live`) durante a sessão.


In [ ]:
%%capture
!pip install -q gradio


In [ ]:
import json, subprocess, gradio as gr, os

def gerar(topico, titulo, aspect, provider, model, api_key, style,
          tts_chain, voice, yt_clips, yt_auto,
          transitions, td, sub_style, mood, chars, subsize,
          particulas, brilho, vineta, cores, barra, rotulo):
    cfg = {"topic": topico, "aspect": aspect, "provider": provider,
           "model": model or None, "api_key": api_key or None, "script_style": style,
           "tts_providers": [x.strip() for x in tts_chain.split(",") if x.strip()],
           "elevenlabs_voice": voice,
           "yt_clips": [l.strip() for l in yt_clips.splitlines() if l.strip()] or None,
           "yt_auto": yt_auto,
           "transitions": transitions, "transition_duration": td,
           "subtitle_style": sub_style, "music_mood": mood,
           "subtitle_max_chars": int(chars), "subtitle_size": int(subsize),
           "particles": particulas, "shine": brilho, "vignette": vineta,
           "color_boost": cores, "progress_bar": barra,
           "output_name": "video_final.mp4"}
    if titulo.strip():
        cfg["title"] = titulo
    if rotulo.strip():
        cfg["scenes"] = [l.strip() for l in rotulo.splitlines() if l.strip()]
    else:
        cfg["n_scenes"] = 5
    json.dump(cfg, open("config.json", "w"), ensure_ascii=False)
    p = subprocess.run(["python", "main.py", "--config", "config.json"], capture_output=True, text=True)
    if p.returncode != 0:
        raise gr.Error("falhou: " + p.stderr[-300:])
    return "out/video_final.mp4", p.stdout[-500:]

iface = gr.Interface(
    fn=gerar,
    inputs=[gr.Textbox(label="Tema", value="Fatos incríveis sobre o universo"),
            gr.Textbox(label="Título (opcional)"),
            gr.Radio(["9:16", "16:9", "1:1"], value="9:16", label="Formato"),
            gr.Radio(["auto", "template", "openrouter", "huggingface", "gemini"],
                     value="auto", label="Roteiro (com fallbacks)"),
            gr.Textbox(label="Modelo (opcional)"),
            gr.Textbox(label="API key (opcional)", type="password"),
            gr.Radio(["viral", "educativo", "storytelling"], value="viral", label="Estilo do roteiro"),
            gr.Textbox(label="Cadeia de voz (com fallbacks)", value="elevenlabs,piper"),
            gr.Radio(["rachel", "drew", "bella", "gabriel", "michele"], value="rachel", label="Voz ElevenLabs"),
            gr.Textbox(lines=3, label="Clipes do YouTube (linha por cena: URL|segundo ou busca|segundo)"),
            gr.Checkbox(False, label="Buscar clipes do YouTube automaticamente"),
            gr.Textbox(label="Transições", value="fade,slideleft,zoomin,smoothleft,circleopen"),
            gr.Slider(0.1, 1.2, value=0.5, step=0.1, label="Duração da transição (s)"),
            gr.Radio(["pop", "box", "clean"], value="pop", label="Estilo das legendas"),
            gr.Radio(["calm", "energetic"], value="calm", label="Música"),
            gr.Slider(18, 44, value=26, step=1, label="Máx. caracteres da legenda"),
            gr.Slider(40, 96, value=64, step=4, label="Tamanho da legenda"),
            gr.Checkbox(True, label="Partículas"),
            gr.Checkbox(True, label="Brilho (shine)"),
            gr.Checkbox(True, label="Vinheta"),
            gr.Checkbox(True, label="Realce de cores"),
            gr.Checkbox(True, label="Barra de progresso"),
            gr.Textbox(lines=4, label="ROTULO (uma fala por linha; vazio = automático)")],
    outputs=[gr.Video(label="Vídeo gerado"), gr.Textbox(label="Log")],
    title="🎬 VideoFX",
)
iface.launch(share=True, debug=True)


## ℹ️ Como funciona — cadeias de fallback

Nada disso deixa o vídeo "tosco": cada camada tenta o melhor recurso disponível e degrada com elegância:

| Camada | Cadeia (na ordem) |
|---|---|
| **Roteiro** | Gemini → OpenRouter → HuggingFace → template local |
| **Voz** | ElevenLabs (multilingual v2) → Piper pt-BR neural |
| **Imagens** | arquivo local → `image_urls` → Pollinations (FLUX) → AI Horde → Pixabay (acervo) → fundo procedural |
| **Cenas** | trecho do YouTube (yt-dlp) → imagens IA → procedural |

Implementação: [github.com/Jushirambo/videofx](https://github.com/Jushirambo/videofx)
